In [ ]:
# ── LOADING ORDER ─────────────────────────────────────────────────────────────
# 1. Communes (geometry)  
# 2. Socioeco (Filosofi + Census)
# 3. BPE Facilities
# 4. Accessibility grid
# ─────────────────────────────────────────────────────────────────────────────

In [2]:
from sqlalchemy import create_engine, text
import sys
sys.path.insert(0, r"C:\Users\admin\work\idf_project\notebooks")
import config as cfg

engine = create_engine(cfg.DB_URL)

# Quick connection test
with engine.connect() as conn:
    result = conn.execute(text("SELECT COUNT(*) FROM information_schema.tables WHERE table_schema = 'idf'"))
    print(f"Connected. Tables in idf schema: {result.scalar()}")

Connected. Tables in idf schema: 6


In [3]:
import geopandas as gpd
from sqlalchemy import text

# Read communes from GeoPackage, filtering to IDF
communes = gpd.read_file(
    cfg.ADMIN_PATH,
    layer='commune',
    where="code_insee_de_la_region = '11'"
)

print(f"Shape: {communes.shape}")
print(f"CRS: {communes.crs}")
print(f"Geometry type: {communes.geom_type.unique()}")
print(communes[['code_insee', 'nom_officiel', 'code_insee_du_departement', 'code_insee_de_la_region']].head(3))

Shape: (1266, 17)
CRS: EPSG:2154
Geometry type: <ArrowStringArray>
['MultiPolygon']
Length: 1, dtype: str
  code_insee      nom_officiel code_insee_du_departement  \
0      75056             Paris                        75   
1      77001  Achères-la-Forêt                        77   
2      77002           Amillis                        77   

  code_insee_de_la_region  
0                      11  
1                      11  
2                      11  


In [ ]:
# Keep only columns we need
communes = communes[[
    'code_insee',
    'nom_officiel', 
    'code_insee_du_departement',
    'code_insee_de_la_region',
    'geometry'
]].rename(columns={
    'code_insee':                  'insee_com',
    'nom_officiel':                'nom_com',
    'code_insee_du_departement':   'insee_dep',
    'code_insee_de_la_region':     'insee_reg',
})

# Make sure CRS is explicitly set to Lambert-93
communes = communes.set_crs('EPSG:2154', allow_override=True)

# Drop existing table and reload
with engine.connect() as conn:
    conn.execute(text("DROP TABLE IF EXISTS idf.communes CASCADE"))
    conn.commit()

# Load to PostGIS
communes.to_postgis(
    'communes',
    engine,
    schema='idf',
    if_exists='replace',
    index=False
)

# Verify
with engine.connect() as conn:
    result = conn.execute(text("""
        SELECT 
            COUNT(*) as total,
            ST_AsText(ST_Centroid(ST_Union(geom))) as idf_centroid,
            ST_SRID(geom) as srid
        FROM idf.communes
    """))
    row = result.fetchone()
    print(f"Communes loaded: {row[0]}")
    print(f"IDF centroid: {row[1]}")
    print(f"SRID: {row[2]}")

In [5]:
import pandas as pd
from sqlalchemy import text
import geopandas as gpd
import sys
sys.path.insert(0, r"C:\Users\admin\work\idf_project\notebooks")
import config as cfg
from sqlalchemy import create_engine

engine = create_engine(cfg.DB_URL)

# Load Filosofi
filo = pd.read_csv(
    cfg.FILO_PATH,
    sep=';',
    dtype=str,
    low_memory=False
)

print(f"Raw shape: {filo.shape}")
print(f"Columns: {filo.columns.tolist()}")
print(filo.head(3))

Raw shape: (833400, 9)
Columns: ['GEO', 'GEO_OBJECT', 'FILOSOFI_MEASURE', 'UNIT_MEASURE', 'UNIT_MULT', 'CONF_STATUS', 'OBS_STATUS', 'TIME_PERIOD', 'OBS_VALUE']
     GEO GEO_OBJECT      FILOSOFI_MEASURE UNIT_MEASURE UNIT_MULT CONF_STATUS  \
0  14226        COM          S_SOC_BEN_DI           PT         0           C   
1  14215        COM          S_SOC_BEN_DI           PT         0           F   
2  14305        COM  S_SOC_BEN_DI_MIN_SOC           PT         0           C   

  OBS_STATUS TIME_PERIOD OBS_VALUE  
0          A        2021       NaN  
1          A        2021       4.9  
2          A        2021       NaN  


In [3]:
nomen = pd.read_csv(
    cfg.RAW / 'bpe' / 'BPE24_table_passage.csv',
    sep=';',
    dtype=str,
    encoding='utf-8'
)

In [4]:
sample = gpd.read_file(cfg.ADMIN_PATH, layer='commune', rows=2)
print(sample.dtypes)

cleabs                                       str
nom_officiel                                 str
nom_officiel_en_majuscules                   str
statut                                       str
code_insee                                   str
population                                 int32
date_du_recensement               datetime64[ms]
organisme_recenseur                          str
code_insee_du_canton                         str
code_insee_de_l_arrondissement               str
code_insee_du_departement                    str
code_insee_de_la_region                      str
code_siren                                   str
codes_siren_des_epci                         str
code_postal                                  str
superficie_cadastrale                      int32
geometry                                geometry
dtype: object


In [5]:
# Filter to commune level only
filo_com = filo[filo['GEO_OBJECT'] == 'COM'].copy()

# Keep only the measures we need
measures_keep = ['MED_SL', 'PR_MD60', 'D1_SL', 'D9_SL', 'IR_D9_D1_SL']
filo_com = filo_com[filo_com['FILOSOFI_MEASURE'].isin(measures_keep)]

# Drop rows with no value
filo_com = filo_com.dropna(subset=['OBS_VALUE'])

# Pivot to wide format — one row per commune, one column per measure
filo_wide = filo_com.pivot_table(
    index='GEO',
    columns='FILOSOFI_MEASURE',
    values='OBS_VALUE',
    aggfunc='first'
).reset_index()

# Flatten column names
filo_wide.columns.name = None

# Rename to our schema
filo_wide = filo_wide.rename(columns={
    'GEO':          'insee_com',
    'MED_SL':       'med_income_2021',
    'PR_MD60':      'poverty_rate',
    'D1_SL':        'd1_income',
    'D9_SL':        'd9_income',
    'IR_D9_D1_SL':  'gini'   # actually D9/D1 ratio, not Gini — we'll note this
})

# Filter to IDF
filo_idf = filo_wide[filo_wide['insee_com'].str[:2].isin(cfg.IDF_DEPS)].copy()

# Convert values to numeric
for col in ['med_income_2021', 'poverty_rate', 'd1_income', 'd9_income', 'gini']:
    filo_idf[col] = pd.to_numeric(filo_idf[col], errors='coerce')

print(f"IDF communes with Filosofi data: {len(filo_idf)}")
print(f"Missing values per column:")
print(filo_idf.isnull().sum())
print(filo_idf.head())

IDF communes with Filosofi data: 1251
Missing values per column:
insee_com            0
d1_income          709
d9_income          709
gini               709
med_income_2021      0
poverty_rate       785
dtype: int64
      insee_com  d1_income  d9_income  gini  med_income_2021  poverty_rate
25944     75056    11240.0    71610.0   6.4            29730          16.0
26638     77001        NaN        NaN   NaN            30970           NaN
26639     77002        NaN        NaN   NaN            26310           NaN
26640     77003        NaN        NaN   NaN            28400           NaN
26641     77004        NaN        NaN   NaN            28400           NaN


In [6]:
from sqlalchemy import VARCHAR, Numeric

filo_idf.to_sql(
    'socioeco',
    engine,
    schema='idf',
    if_exists='replace',
    index=False,
    dtype={
        'insee_com':       VARCHAR(5),
        'med_income_2021': Numeric,
        'poverty_rate':    Numeric,
        'd1_income':       Numeric,
        'd9_income':       Numeric,
        'gini':            Numeric
    }
)

# Verify
with engine.connect() as conn:
    result = conn.execute(text("""
        SELECT 
            COUNT(*) as total,
            COUNT(med_income_2021) as has_income,
            COUNT(poverty_rate) as has_poverty,
            ROUND(AVG(med_income_2021::numeric)) as avg_median_income
        FROM idf.socioeco
    """))
    row = result.fetchone()
    print(f"Total communes: {row[0]}")
    print(f"With median income: {row[1]}")
    print(f"With poverty rate: {row[2]}")
    print(f"Average median income IDF: {row[3]}€")

Total communes: 1251
With median income: 1251
With poverty rate: 466
Average median income IDF: 27353€


In [7]:
census = pd.read_csv(
    cfg.CENSUS_PATH,
    sep=';',
    dtype=str,
    low_memory=False
)

print(f"Shape: {census.shape}")
print(f"Columns: {census.columns.tolist()}")
print(census.head(3))

Shape: (49276, 76)
Columns: ['IRIS', 'COM', 'TYP_IRIS', 'LAB_IRIS', 'P22_POP', 'P22_POP0002', 'P22_POP0305', 'P22_POP0610', 'P22_POP1117', 'P22_POP1824', 'P22_POP2539', 'P22_POP4054', 'P22_POP5564', 'P22_POP6579', 'P22_POP80P', 'P22_POP0014', 'P22_POP1529', 'P22_POP3044', 'P22_POP4559', 'P22_POP6074', 'P22_POP75P', 'P22_POP0019', 'P22_POP2064', 'P22_POP65P', 'P22_POPH', 'P22_H0014', 'P22_H1529', 'P22_H3044', 'P22_H4559', 'P22_H6074', 'P22_H75P', 'P22_H0019', 'P22_H2064', 'P22_H65P', 'P22_POPF', 'P22_F0014', 'P22_F1529', 'P22_F3044', 'P22_F4559', 'P22_F6074', 'P22_F75P', 'P22_F0019', 'P22_F2064', 'P22_F65P', 'C22_POP15P', 'C22_POP15P_STAT_GSEC11_21', 'C22_POP15P_STAT_GSEC12_22', 'C22_POP15P_STAT_GSEC13_23', 'C22_POP15P_STAT_GSEC14_24', 'C22_POP15P_STAT_GSEC15_25', 'C22_POP15P_STAT_GSEC16_26', 'C22_POP15P_STAT_GSEC32', 'C22_POP15P_STAT_GSEC40', 'C22_H15P', 'C22_H15P_STAT_GSEC11_21', 'C22_H15P_STAT_GSEC12_22', 'C22_H15P_STAT_GSEC13_23', 'C22_H15P_STAT_GSEC14_24', 'C22_H15P_STAT_GSEC15_25'

In [8]:
# Select only columns we need
census_cols = list(cfg.CENSUS_KEEP.keys())
census_sub = census[census_cols].copy()

# Convert to numeric (currently strings)
for col in census_cols:
    if col != 'COM':
        census_sub[col] = pd.to_numeric(census_sub[col], errors='coerce')

# Filter to IDF communes
census_sub = census_sub[census_sub['COM'].str[:2].isin(cfg.IDF_DEPS)]

# Aggregate from IRIS to commune level by summing
census_com = census_sub.groupby('COM').sum().reset_index()

# Rename columns to our schema
census_com = census_com.rename(columns=cfg.CENSUS_KEEP)

# Compute derived indicators
census_com['pct_65plus'] = (
    census_com['pop_65plus'] / census_com['pop_total'] * 100
).round(2)

census_com['pct_0014'] = (
    census_com['pop_0014'] / census_com['pop_total'] * 100
).round(2)

# Round population columns to integers
for col in ['pop_total', 'pop_65plus', 'pop_0014', 'pop_immigrant', 'pop_households']:
    census_com[col] = census_com[col].round(0).astype('Int64')

print(f"IDF communes in census: {len(census_com)}")
print(f"Missing values:\n{census_com.isnull().sum()}")
print(f"\nTotal IDF population: {census_com['pop_total'].sum():,.0f}")
print(census_com.head())

IDF communes in census: 1286
Missing values:
insee_com         0
pop_total         0
pop_65plus        0
pop_0014          0
pop_immigrant     0
pop_households    0
pct_65plus        0
pct_0014          0
dtype: int64

Total IDF population: 12,380,964
  insee_com  pop_total  pop_65plus  pop_0014  pop_immigrant  pop_households  \
0     75101      15475        2749      1489           2882           15221   
1     75102      20433        2380      2382           3735           20227   
2     75103      32772        5254      3391           6612           32460   
3     75104      28039        4992      3010           5285           27543   
4     75105      55925       10935      5939           9632           53552   

   pct_65plus  pct_0014  
0       17.76      9.62  
1       11.65     11.66  
2       16.03     10.35  
3       17.80     10.74  
4       19.55     10.62  


In [9]:
# Merge Filosofi and Census into one dataframe first
socioeco = census_com.merge(
    filo_idf,
    on='insee_com',
    how='left'  # keep all census communes, attach Filosofi where available
)

print(f"Combined shape: {socioeco.shape}")
print(f"Columns: {socioeco.columns.tolist()}")
print(f"Missing Filosofi data: {socioeco['med_income_2021'].isnull().sum()} communes")

# Load combined table
from sqlalchemy import Integer, Float

socioeco.to_sql(
    'socioeco',
    engine,
    schema='idf',
    if_exists='replace',
    index=False,
    dtype={
        'insee_com':       VARCHAR(5),
        'pop_total':       Integer,
        'pop_65plus':      Integer,
        'pop_0014':        Integer,
        'pop_immigrant':   Integer,
        'pop_households':  Integer,
        'pct_65plus':      Numeric,
        'pct_0014':        Numeric,
        'med_income_2021': Numeric,
        'poverty_rate':    Numeric,
        'd1_income':       Numeric,
        'd9_income':       Numeric,
        'gini':            Numeric,
    }
)

# Verify
with engine.connect() as conn:
    result = conn.execute(text("""
        SELECT 
            COUNT(*) as total,
            COUNT(med_income_2021) as has_income,
            COUNT(poverty_rate) as has_poverty,
            ROUND(AVG(pop_total)) as avg_population
        FROM idf.socioeco
    """))
    row = result.fetchone()
    print(f"Total communes: {row[0]}")
    print(f"With median income: {row[1]}")
    print(f"With poverty rate: {row[2]}")
    print(f"Avg population per commune: {row[3]}")

Combined shape: (1286, 13)
Columns: ['insee_com', 'pop_total', 'pop_65plus', 'pop_0014', 'pop_immigrant', 'pop_households', 'pct_65plus', 'pct_0014', 'd1_income', 'd9_income', 'gini', 'med_income_2021', 'poverty_rate']
Missing Filosofi data: 36 communes
Total communes: 1286
With median income: 1250
With poverty rate: 465
Avg population per commune: 9627


In [ ]:

from shapely.geometry import Point

# Load full BPE, filter to IDF and our target facility types only
bpe = pd.read_csv(
    cfg.BPE_PATH,
    sep=';',
    dtype=str,
    low_memory=False
)

# Rename columns
bpe = bpe.rename(columns=cfg.BPE_RENAME)

# Filter to IDF
bpe_idf = bpe[bpe['dep'].isin(cfg.IDF_DEPS)].copy()

# Filter to only facility types we care about
target_codes = list(cfg.FACILITY_TYPES.keys())
bpe_idf = bpe_idf[bpe_idf['typequ'].isin(target_codes)].copy()

# Add human readable label and indicator name from config
bpe_idf['typequ_label'] = bpe_idf['typequ'].map(
    dict(zip(nomen['TYPEQU'], nomen['Libelle_TYPEQU']))
)
bpe_idf['indicator'] = bpe_idf['typequ'].map(cfg.FACILITY_TYPES)

# Drop rows with missing coordinates
bpe_idf = bpe_idf.dropna(subset=['X', 'Y'])
bpe_idf['X'] = bpe_idf['X'].astype(float)
bpe_idf['Y'] = bpe_idf['Y'].astype(float)

print(f"IDF facilities (target types only): {len(bpe_idf)}")
print(f"\nFacilities by indicator:")
print(bpe_idf['indicator'].value_counts().to_string())

In [ ]:
# Build geometry from X/Y coordinates (already in Lambert-93 / EPSG:2154)
geometry = [Point(xy) for xy in zip(bpe_idf['X'], bpe_idf['Y'])]
gdf = gpd.GeoDataFrame(
    bpe_idf[['typequ', 'typequ_label', 'indicator', 'dep', 'insee_com']],
    geometry=geometry,
    crs='EPSG:2154'
)

print(f"GeoDataFrame shape: {gdf.shape}")
print(f"CRS: {gdf.crs}")
print(gdf.head(3))

# Push to PostGIS
gdf.to_postgis(
    'facilities',
    engine,
    schema='idf',
    if_exists='replace',
    index=False
)

# Verify
with engine.connect() as conn:
    result = conn.execute(text("""
        SELECT 
            COUNT(*) as total,
            COUNT(DISTINCT typequ) as unique_types,
            COUNT(DISTINCT dep) as deps
        FROM idf.facilities
    """))
    row = result.fetchone()
    print(f"\nLoaded: {row[0]} facilities")
    print(f"Unique equipment types: {row[1]}")
    print(f"Départements: {row[2]}")

In [12]:
import duckdb

# Preview what we're working with
summary = duckdb.sql(f"""
    SELECT 
        COUNT(*) as total_rows,
        COUNT(DISTINCT depcom) as communes,
        COUNT(DISTINCT typeeq_id) as equipment_types,
        MIN(duree) as min_travel_min,
        MAX(duree) as max_travel_min,
        AVG(duree) as avg_travel_min
    FROM read_parquet('{cfg.ACCESS_PATH.as_posix()}')
    WHERE typeeq_id IN ({','.join([f"'{c}'" for c in target_codes])})
""").df()

print(summary.to_string())

   total_rows  communes  equipment_types  min_travel_min  max_travel_min  avg_travel_min
0     5449320      1267               60             1.0           94.86        9.613685


In [13]:
# Aggregate to commune level — weighted average travel time per commune per equipment type
# We weight by grid cell population so that populous areas within a commune
# contribute more to the commune's average than empty fields
access_agg = duckdb.sql(f"""
    SELECT 
        depcom AS insee_com,
        typeeq_id,
        -- Population-weighted average travel time
        SUM(duree * pop) / NULLIF(SUM(pop), 0) AS avg_travel_min,
        -- Simple average as cross-check
        AVG(duree) AS simple_avg_min,
        -- Population covered
        SUM(pop) AS total_pop,
        -- Number of grid cells
        COUNT(*) AS n_cells
    FROM read_parquet('{cfg.ACCESS_PATH.as_posix()}')
    WHERE typeeq_id IN ({','.join([f"'{c}'" for c in target_codes])})
    AND pop > 0  -- exclude unpopulated grid cells
    GROUP BY depcom, typeeq_id
    ORDER BY depcom, typeeq_id
""").df()

print(f"Shape: {access_agg.shape}")
print(f"Communes: {access_agg['insee_com'].nunique()}")
print(f"Equipment types: {access_agg['typeeq_id'].nunique()}")
print(f"\nSample — Paris travel times:")
paris = access_agg[access_agg['insee_com'] == '75056'].sort_values('avg_travel_min')
print(paris[['typeeq_id','avg_travel_min','simple_avg_min']].to_string())

Shape: (76020, 6)
Communes: 1267
Equipment types: 60

Sample — Paris travel times:
   typeeq_id  avg_travel_min  simple_avg_min
9       B207        1.076762        1.124839
8       B202        1.081546        1.133175
39      D307        1.114525        1.177935
35      D265        1.116882        1.184587
41      D502        1.182306        1.279841
6       B105        1.228730        1.297435
42      D505        1.285078        1.386423
36      D277        1.294440        1.327175
37      D281        1.377431        1.477211
10      C107        1.489515        1.626828
29      C610        1.495725        1.530286
7       B201        1.512241        1.574629
12      C109        1.579573        1.709197
32      D108        1.631980        1.735036
55      F121        1.691219        1.793801
38      D302        1.717928        1.791785
13      C201        1.732714        1.805328
54      F120        1.756608        1.796161
2       A206        1.772088        1.846781
51      F111     

In [15]:
# Load aggregated accessibility data to PostGIS
access_agg.to_sql(
    'accessibility_grid',
    engine,
    schema='idf',
    if_exists='replace',
    index=False,
    dtype={
        'insee_com':      VARCHAR(5),
        'typeeq_id':      VARCHAR(10),
        'avg_travel_min': Numeric,
        'simple_avg_min': Numeric,
        'total_pop':      Numeric,
        'n_cells':        Integer,
    }
)

# Add indexes for fast querying in Phase 5
with engine.connect() as conn:
    conn.execute(text("""
        CREATE INDEX IF NOT EXISTS idx_access_commune 
        ON idf.accessibility_grid(insee_com);
        
        CREATE INDEX IF NOT EXISTS idx_access_type 
        ON idf.accessibility_grid(typeeq_id);
        
        CREATE INDEX IF NOT EXISTS idx_access_commune_type 
        ON idf.accessibility_grid(insee_com, typeeq_id);
    """))
    conn.commit()

# Verify
with engine.connect() as conn:
    result = conn.execute(text("""
        SELECT 
            COUNT(*) as total_rows,
            COUNT(DISTINCT insee_com) as communes,
            COUNT(DISTINCT typeeq_id) as equipment_types,
            ROUND(AVG(avg_travel_min)::numeric, 2) as overall_avg_min
        FROM idf.accessibility_grid
    """))
    row = result.fetchone()
    print(f"Rows: {row[0]}")
    print(f"Communes: {row[1]}")
    print(f"Equipment types: {row[2]}")
    print(f"Overall avg travel time: {row[3]} min")

Rows: 76020
Communes: 1267
Equipment types: 60
Overall avg travel time: 12.48 min
